<img src="MUML_header.svg" alt="Header" style="width:100%; height:auto;">

# 01_02 Naive Bayes

In this notebook you will:

- Revisit scikit-learn.
- Learn Naive Bayes:
  - GaussianNB for continuous features.
  - MultinomialNB for discrete counts.
- Use `predict_proba` to inspect class probabilities.
- Plot histograms per class for intuition.
- Introduce `train_test_split` to compare training vs test performance.
- Convert raw text to features with `CountVectorizer`.

## Gaussian Naive Bayes on Wine dataset

### Step 1: Load the Wine dataset and keep NumPy arrays

- Use [`load_wine()`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_wine.html) to load the Wine dataset.
- Prepare:
  - `X` as a NumPy array of shape `(n_samples, n_features)`.
  - `y` as integer class ids (0,1,2).
  - `y_names` as string class names (you can use `wine.target_names`).


In [21]:
import pandas as pd
from sklearn.datasets import load_wine
wine = load_wine(as_frame=True)
df = wine.frame
print(df.info())
print(df.head())

X = df[:-1]
print(X.head())
y = df['target']
y_names = wine.target_names


<class 'pandas.DataFrame'>
RangeIndex: 178 entries, 0 to 177
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   alcohol                       178 non-null    float64
 1   malic_acid                    178 non-null    float64
 2   ash                           178 non-null    float64
 3   alcalinity_of_ash             178 non-null    float64
 4   magnesium                     178 non-null    float64
 5   total_phenols                 178 non-null    float64
 6   flavanoids                    178 non-null    float64
 7   nonflavanoid_phenols          178 non-null    float64
 8   proanthocyanins               178 non-null    float64
 9   color_intensity               178 non-null    float64
 10  hue                           178 non-null    float64
 11  od280/od315_of_diluted_wines  178 non-null    float64
 12  proline                       178 non-null    float64
 13  target          

### Step 2: Train a Gaussian Naive Bayes model

- Create an instance of `GaussianNB()`.
- Train the model with the data
- Print the predicted classes for the data
- Print the learned class labels via `.classes_`.
- Compute the percentage of correctly classified instances on the same data


In [ ]:
impo
gNB = GaussianNB()

### Step 3: Use `predict_proba` to see class probabilities
 So far, we used `.predict(X)` which gives the **most likely class** for each instance.  
- With `.predict_proba(X)` we can inspect the **full probability distribution over all classes**.  
  - Each row corresponds to one sample.  
  - Each column corresponds to one class.  
  - Values are probabilities and **sum to 1 across a row**.  
- This is useful when we want to know not only the predicted class, but also **how confident** the model is.  

**What to do:**
1. Call `.predict_proba(X)` and print the first few rows.  
2. Verify that row sums are equal to 1.  
3. Compare it with `.predict(X)` by checking that the predicted label corresponds to the class with the **highest probability** in the row.  



### Step 4: Inspect the parameters learned by GaussianNB

When we train a Gaussian Naive Bayes model, it estimates simple statistical parameters for each class and feature:  

- **`.class_prior_`** → the prior probability of each class, estimated from the relative frequencies of the training data (unless specified manually).  
- **`.theta_`** → the mean of each feature within each class. Each row corresponds to a class, and each column to a feature.  
- **`.var_`** → the variance of each feature within each class. Together with the means, this defines the Gaussian distribution used for that feature in that class.  

These parameters allow GaussianNB to compute the probability of a sample belonging to a class by multiplying the Gaussian likelihoods of its features (assuming independence).  

**Note:** if a feature is actually discrete (e.g., categorical encoded as integers), GaussianNB will still treat it as if it came from a continuous Gaussian distribution. In practice this may be a poor fit, but the model still produces means and variances. If the data is truly categorical, a different variant such as `CategoricalNB` or `MultinomialNB` should be used.  

**What to do:**
1. Print `.class_prior_`.  
2. Wrap `.theta_` into a DataFrame for readability (rows = classes, columns = features).  
3. Do the same with `.var_`.  
4. Inspect these tables to see the per-class means and variances.  



### Step 5: Evaluate generalization with a train/test split

So far, we have measured the model’s performance on the same data it was trained on.  
This training accuracy is usually over-optimistic because the model has already seen those examples.  
To get a more honest estimate of how the model generalizes to unseen data, we split our dataset into two parts:  

- **Training set** → used to fit the model parameters.  
- **Test set** → held out and used only to evaluate performance after training.  

In scikit-learn, this is done with the function `train_test_split`. Important arguments are:  

- `test_size` → fraction of data to put in the test set (e.g. `0.3` for 30%).  
- `random_state` → a seed to make the split reproducible.  
- `stratify=y` → ensures that class proportions are similar in both train and test sets (very important for classification).  

**What to do:**  
1. Import `train_test_split` from `sklearn.model_selection`.  
2. Split `X` and `y` into `X_train, X_test, y_train, y_test` using a 70/30 split, with `random_state=42` and `stratify=y`.  
3. Fit a new `GaussianNB` model on the training set.  
4. Print training accuracy and test accuracy with `.score()`.  


### Step 6: Plot histograms of selected features by class

Now we want to look at the distribution of specific features (e.g., `"alcohol"` or `"color_intensity"`) for each class.  
This lets us see how separable the classes are based on individual features, and also check how well the Gaussian assumption might fit.  

- Plot two histograms, one for each of the feature values (`"alcohol"` or `"color_intensity"`) for each class on the same axes, using transparency so the distributions can be compared.


## MultinomialNB for text

Now we move to **text data**, which requires a different representation:  

- Raw text (documents, articles, emails, …) cannot be used directly by machine learning models.  
- A simple and very common approach is the **bag-of-words** model:  
  - We build a vocabulary of all words that appear in the corpus.  
  - Each document is then represented as a vector of **word counts** (how many times each word appears).  
  - This is done automatically with scikit-learn’s `CountVectorizer`.  

The result is a large but sparse matrix, where:  
- Rows = documents  
- Columns = words in the vocabulary  
- Entries = counts of that word in that document

For this kind of data, Naive Bayes with a Multinomial distribution (`MultinomialNB`) is a natural choice because it models features as discrete counts.  

### Step 1: Load a text dataset and build a bag-of-words representation

We will use a subset of the **20 Newsgroups dataset**, a collection of Usenet newsgroup posts on different topics.  
It is a standard benchmark for text classification in scikit-learn.  
To keep things simple, we’ll select only **two categories** (e.g. `"rec.sport.hockey"` and `"sci.space"`).  

**What to do:**  
1. Import `fetch_20newsgroups` from `sklearn.datasets`.

 ~~~
    news_train = fetch_20newsgroups(
    subset="train",
    categories=["rec.sport.hockey", "sci.space"],
    remove=("headers", "footers", "quotes"))

3. Load the training subset with two chosen categories.  
4. Store the raw text documents in `docs` and their labels in `y_text`.  
5. Inspect the class names via `target_names`.  

### Step 2: Convert text documents into a bag-of-words matrix

Now that we have raw text documents, we need to transform them into numerical features.  
We will use **`CountVectorizer`**, which implements the **bag-of-words** model:

- It builds a **vocabulary** of all words in the dataset (after preprocessing).  
- Each document is converted into a vector of **word counts**.  
- The result is a **document–term matrix** with shape `(n_documents, n_words)`.  

**Note:** By default, this matrix is stored as a **sparse matrix** (`scipy.sparse.csr_matrix`),  
because most words do not appear in most documents. This is normal and efficient.  

**What to do:**  
1. Import `CountVectorizer` from `sklearn.feature_extraction.text`.  
2. Create a vectorizer with reasonable settings, for example:  
   - `stop_words="english"` → ignore common English words like “the”, “and”.  
   - `max_features=4000` → keep at most 4000 words in the vocabulary.  
   - `min_df=2` → ignore words that appear in only 1 document.  
3. Call `.fit_transform(docs)` to build the vocabulary and transform documents into a sparse count matrix `X_text`.  
4. Print the shape of `X_text` (documents × vocabulary size).  
5. Print the first 10 feature names from the vocabulary.  
6. Optionally: convert a small slice of `X_text` to dense with `.toarray()` just to see the counts.  


### Step 3: Train a Multinomial Naïve Bayes model and inspect probabilities

Now that `X_text` is a **sparse bag-of-words matrix** and `y_text` are the labels, we can train a text classifier.

- Create an instance of `MultinomialNB`.
- Train the model on `(X_text, y_text)` (it accepts scipy sparse matrices directly).  

- Use `.predict_proba` to see the class probability distribution for a few documents.
- Compute the training accuracy.


### Step 4: Train/test split for text — check generalization


* Split the bag-of-words matrix `X_text` and labels `y_text` into train/test:
   - `test_size=0.3`, `random_state=42`, `stratify=y_text`.
* Fit a fresh `MultinomialNB` on the training split.
* Report training and test accuracies.



### Step 5: Building a Pipeline for text classification

So far we separated the steps: first used `CountVectorizer` to transform the text,  
then trained `MultinomialNB` on the resulting bag-of-words matrix.  

In practice, it is convenient to chain these steps together into a Pipeline:  
- The raw text is passed directly into the pipeline.  
- The pipeline should first run `CountVectorizer`.  
- Then it should apply `MultinomialNB`.  
- The result behaves just like any other scikit-learn estimator with `.fit`, `.predict`, and `.score`.  

**What to do:**  
1. Import `Pipeline` from `sklearn.pipeline`.  
2. Build a pipeline with two steps:  
   - `"vect"` → `CountVectorizer(stop_words="english", max_features=4000, min_df=2)`  
   - `"clf"`  → `MultinomialNB(alpha=1.0)`   
3. Fit the pipeline on the training data.  
4. Print training and test accuracy.  
5. Try changing the vectorizer settings (e.g. `ngram_range`, `binary=True`, `min_df`) and see how performance changes.  
